In [0]:
from pyspark.sql.types import StructType, StructField,IntegerType, StringType,BooleanType, DoubleType,LongType,TimestampType
import pyspark.sql.functions as F


In [0]:
# Reading the file as text
# This will create a column with the name "value"
df = spark.read.text("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Product.csv")

df = df.withColumnRenamed("value", "raw_data")

# Displaying some raw_data records to see the separator is a tab character(\t)
df.show(5, truncate=False)

In [0]:
df = spark.read.csv("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Product.csv", sep="\t", header=True)


df.show(5, truncate=False)

In [0]:
# define the schema
schema = StructType([
    StructField("product_key",IntegerType(), True),
    StructField("product",StringType(), True),
    StructField("standard_cost",StringType(), True),
    StructField("color",StringType(), True),
    StructField("subcategory",StringType(), True),
    StructField("category",StringType(), True),
    StructField("background_color_format",StringType(), True),
    StructField("font_color_format",StringType(),True)

])

df1 = spark.read.schema(schema).csv("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Product.csv", sep="\t",header=True)
display(df1)

In [0]:


df_silver = (
    df1.dropDuplicates()
    #1.αφαιρεση των whitespace
    .withColumn("product", F.trim(F.col("product")))
    .withColumn("color", F.trim(F.col("color")))
    .withColumn("subcategory", F.trim(F.col("subcategory")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("background_color_format", F.trim(F.col("background_color_format")))
    .withColumn("font_color_format", F.trim(F.col("font_color_format")))
    .withColumn("standard_cost", F.trim(F.col("standard_cost")))
    # 2. ΚΑΘΑΡΙΣΜΟΣ & CASTING: Βγάζουμε το $ και το , και το κάνουμε double
    .withColumn("standard_cost", F.regexp_replace("standard_cost", r"[\$,]", "").cast("double"))
    # 3. FILTER (Outliers): ΤΩΡΑ που το StandardCost είναι πια καθαρός αριθμός (double), 
    # μπορούμε να κάνουμε μαθηματικές συγκρίσεις (μεγαλύτερο του 0)
    .filter(F.col("standard_cost") > 0) # removing negative/zero standard cost values
    # .filter(F.col("standard_cost") < 5000)
)

# Εμφάνιση του αποτελέσματος στο Databricks
display(df_silver)

In [0]:
%sql
DROP TABLE IF EXISTS accenture_final_project.silver_Product;

## Export as a delta table

In [0]:
# We save the DataFrame as a delta table with name "silver_Product"
df_silver.write.format("delta").mode("overwrite").saveAsTable("accenture_final_project.silver_Product")
